In [6]:
import ee
ee.Authenticate(auth_mode="localhost")

True

In [7]:
import geemap

In [8]:
ee.Initialize(project="change-detection-sentinel2")
print("Initialized")

Initialized


In [9]:
Map = geemap.Map()
print("Map created")

Map created


In [10]:
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [12]:
# Load the FAO GAUL Level 1 administrative boundaries
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")

# Filter the dataset to only Kano State
kano = gaul.filter(ee.Filter.eq("ADM1_NAME", "Kano"))

# Create a new map
Map = geemap.Map()

# Center the map on Kano
Map.centerObject(kano, 8)

# Add the Kano boundary to the map
Map.addLayer(
    kano,
    {"color": "red"},
    "Kano Boundary"
)

# Display the map
Map

EEException: Caller does not have required permission to use project change-detection-sentinel2. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=change-detection-sentinel2 and then retry. Propagation of the new permission may take a few minutes.

In [13]:
print(kano.getInfo())

EEException: Caller does not have required permission to use project change-detection-sentinel2. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=change-detection-sentinel2 and then retry. Propagation of the new permission may take a few minutes.

In [3]:
geemap.ee_export_vector(
    kano,
    filename="Kano_Boundary.geojson"
)

NameError: name 'geemap' is not defined

In [14]:
import ee
import geemap

# Initialize Earth Engine
ee.Initialize(project="change-detection-sentinel2")

# Load the SRTM Digital Elevation Model (DEM)
dem = ee.Image("USGS/SRTMGL1_003")

# Compute slope from the DEM
slope = ee.Terrain.slope(dem)

# Visualization parameters
dem_vis = {
    "min": 0,
    "max": 3000,
    "palette": [
        "006633",  # Dark Green
        "66cc66",  # Green
        "ffff99",  # Yellow
        "cc9966",  # Brown
        "ffffff"   # White
    ]
}

slope_vis = {
    "min": 0,
    "max": 60,
    "palette": [
        "ffffff",  # White
        "ffff00",  # Yellow
        "ff9900",  # Orange
        "ff0000"   # Red
    ]
}

# Create the map
Map = geemap.Map()

# Center the map on Nigeria
Map.setCenter(8.6753, 9.0820, 6)

# Add DEM and Slope layers
Map.addLayer(dem, dem_vis, "SRTM DEM")
Map.addLayer(slope, slope_vis, "Slope")

# Layer control
Map.addLayerControl()

# Display the map
Map

EEException: Caller does not have required permission to use project change-detection-sentinel2. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=change-detection-sentinel2 and then retry. Propagation of the new permission may take a few minutes.

In [ ]:
Map.to_html("dem_slope_map.html")

## Compute Drainage Density using GEE

In [5]:
#import libraries
import ee
import geemap

 # Initialize Earth Engine
ee.Initialize(project="ground-water-geo-mapping")

# Create a map 
Map = geemap.Map( 
    center=[9.0820, 8.6753],  # Center the map on Nigeria
    zoom=6,                  # Set the zoom level
    basemap="HYBRID"         # Set the basemap to HYBRID
)

 # Create area of interest
aoi = ee.Geometry.Rectangle([8.3, 6.7, 9.5, 7.5])    # Adjust coordinates as needed

# load MERIT HYDRO dataset
merit_hydro = ee.Image("MERIT/Hydro/v1_0_1")


#flow accumalation band
flow_accumulation = merit_hydro.select("upa")

#stream extraction
threshold = 100  # Adjust this threshold based on your needs

streams = flow_accumulation.gt(threshold)

#pixel length(metres)
pixel_length = ee.Image.pixelArea().sqrt()

#stream length image
stream_length = streams.multiply(pixel_length)

#Drainage density
area = ee.Image.pixelArea()
drainage_density = stream_length.divide(area)


#visualization
Map.centerObject(aoi, 9)

Map.addLayer(
    flow_accumulation.clip(aoi),
    {
        "min": 0,
        "max": 1000,
        "palette": ["white", "yellow", "orange", "red"]
    },
    "Flow Accumalation"
)

Map.addLayer(
    streams.selfMask().clip(aoi),
    {
        "palette": "blue"
    },
    "Streams"
)

Map.addLayer(
    drainage_density.clip(aoi),
    {
        "min": 0,
        "max": 0.01,
        "palette": ["white", "green", "yellow", "red"]
    },
    "Drainage Density"
)

Map



Map(center=[7.100269362384542, 8.90000000000003], controls=(WidgetControl(options=['position', 'transparent_bg…

## Compute the Topographic Wetness Index (TWI)

In [6]:
import ee
import geemap

# Initialize Earth Engine
ee.Initialize(project="ground-water-geo-mapping")

# Create map
Map = geemap.Map()

# ------------------------------------
# Area of Interest (replace with your AOI)
# ------------------------------------
aoi = ee.Geometry.Rectangle([8.3, 6.7, 9.1, 7.5])


# ------------------------------------
# MERIT Hydro
# ------------------------------------
merit = ee.Image("MERIT/Hydro/v1_0_1")

# Flow accumulation (upstream area)
flow_acc = merit.select("upa")

# ------------------------------------
# DEM
# ------------------------------------
dem = ee.Image("MERIT/DEM/v1_0_3")

# ------------------------------------
# Compute slope
# ------------------------------------
slope = ee.Terrain.slope(dem)

# Convert slope to radians
slope_rad = slope.multiply(3.14159265 / 180)

# ------------------------------------
# Topographic Wetness Index
# TWI = ln(a / tan(beta))
# ------------------------------------

# Avoid division by zero
tan_slope = slope_rad.tan().max(0.001)

twi = flow_acc.divide(tan_slope).log()

# ------------------------------------
# Display
# ------------------------------------
Map.centerObject(aoi, 9)

twi_vis = {
    "min": 0,
    "max": 12,
    "palette": [
        "white",
        "yellow",
        "green",
        "blue"
    ]
}

Map.addLayer(
    twi.clip(aoi),
    twi_vis,
    "Topographic Wetness Index"
)

Map

Map(center=[7.100055299411672, 8.700000000000472], controls=(WidgetControl(options=['position', 'transparent_b…

In [37]:
#Visualization parameters
twi_vis = {
    "min": 0,
    "max": 12,
    "palette": [
        "white",
        "yellow",
        "green",
        "blue"
    ]
}

## Resample Thematic layers to 30m

In [7]:

import ee

ee.Initialize(project="ground-water-geo-mapping")

# Example study area
aoi = ee.Geometry.Rectangle([8.3, 6.7, 9.1, 7.5])

# Elevation (MERIT DEM)
elevation = ee.Image("MERIT/DEM/v1_0_3").clip(aoi)

# Slope from DEM
slope = ee.Terrain.slope(elevation)

# Flow accumulation from MERIT Hydro
merit = ee.Image("MERIT/Hydro/v1_0_1")
flow_acc = merit.select("upa")

# Example drainage density (replace with your class method)
drainage_density = flow_acc.gt(100)

# Example TWI
slope_rad = slope.multiply(3.14159265 / 180)
twi = flow_acc.divide(slope_rad.tan().max(0.001)).log()

# Rainfall (example dataset)
rainfall = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
      .filterDate("2023-01-01", "2023-12-31")
      .sum()
      .clip(aoi)
)

# Land cover (example dataset)
land_cover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(aoi)

# Soil texture
# Replace this with the dataset used in your class
soil_texture = ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02").clip(aoi)

## Export Thematic layers at 30m Spatial Resolution & Clipped to Kano Boundary

In [31]:
#Create kano boundary
import ee

# Initialize Earth Engine
ee.Initialize(project="ground-water-geo-mapping")

#load Nigeria state boundaries
states = ee.FeatureCollection("FAO/GAUL/2015/level1")

#filter the dataset to only Kano State
kano_boundary = states.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Nigeria"),
        ee.Filter.eq("ADM1_NAME", "Kano")

    )
)

In [32]:
import geemap

Map = geemap.Map()
Map.centerObject(kano_boundary, 8)
Map.addLayer(kano_boundary, {"color": "red"}, "Kano Boundary")
Map

Map(center=[11.735699301548118, 8.513584659731869], controls=(WidgetControl(options=['position', 'transparent_…

In [34]:
#Clip Each Resampled Layer to Kano Boundary
rainfall_clip = rainfall.clip(kano_boundary)
elevation_clip = elevation.clip(kano_boundary)
slope_clip = slope.clip(kano_boundary)
land_cover_clip = land_cover.clip(kano_boundary)
soil_texture_clip = soil_texture.clip(kano_boundary)
drainage_density_clip = drainage_density.clip(kano_boundary)
twi_clip = twi.clip(kano_boundary)


#Confirm Resolution is 30 m
rainfall_clip = rainfall_clip.reproject(crs='EPSG:4326', scale=30)
elevation_clip = elevation_clip.reproject(crs='EPSG:4326', scale=30)
slope_clip = slope_clip.reproject(crs='EPSG:4326', scale=30)
land_cover_clip = land_cover_clip.reproject(crs='EPSG:4326', scale=30)
soil_texture_clip = soil_texture_clip.reproject(crs='EPSG:4326', scale=30)
drainage_density_clip = drainage_density_clip.reproject(crs='EPSG:4326', scale=30)
twi_clip = twi_clip.reproject(crs='EPSG:4326', scale=30)

In [38]:
print(kano_boundary)

ee.FeatureCollection({
  "functionInvocationValue": {
    "functionName": "Collection.filter",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.loadTable",
          "arguments": {
            "tableId": {
              "constantValue": "FAO/GAUL/2015/level1"
            }
          }
        }
      },
      "filter": {
        "functionInvocationValue": {
          "functionName": "Filter.and",
          "arguments": {
            "filters": {
              "arrayValue": {
                "values": [
                  {
                    "functionInvocationValue": {
                      "functionName": "Filter.equals",
                      "arguments": {
                        "leftField": {
                          "constantValue": "ADM0_NAME"
                        },
                        "rightValue": {
                          "constantValue": "Nigeria"
                        }
                      }


In [ ]:
#Export function
def export_to_drive(image, description, folder):
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=kano_boundary.geometry(),
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    task.start()
    print(f"{description} export started.")




rainfall_30m export started.
elevation_30m export started.
slope_30m export started.
land_cover_30m export started.
soil_texture_30m export started.
drainage_density_30m export started.
twi_30m export started.


In [42]:
# check rainfall image type
print(type(rainfall))

<class 'ee.image.Image'>


In [43]:

layers = {
    "rainfall_clip": rainfall_clip,
    "elevation_clip": elevation_clip,
    "slope_clip": slope_clip,
    "land_cover_clip": land_cover_clip,
    "soil_texture_clip": soil_texture_clip,
    "drainage_density_clip": drainage_density_clip,
    "twi_clip": twi_clip
}

for name, layer in layers.items():
    print(name, type(layer))

rainfall_clip <class 'ee.image.Image'>
elevation_clip <class 'ee.image.Image'>
slope_clip <class 'ee.image.Image'>
land_cover_clip <class 'ee.image.Image'>
soil_texture_clip <class 'ee.image.Image'>
drainage_density_clip <class 'ee.image.Image'>
twi_clip <class 'ee.image.Image'>


In [44]:
#cancel old READY task
for task in ee.batch.Task.list():
    status = task.status()
    if status['state'] == 'READY':
        task.cancel()
        print("Cancelled:", status['description'])

Cancelled: twi_30m
Cancelled: drainage_density_30m
Cancelled: soil_texture_30m
Cancelled: land_cover_30m
Cancelled: slope_30m
Cancelled: elevation_30m
Cancelled: rainfall_30m
Cancelled: twi_30m
Cancelled: drainage_density_30m
Cancelled: soil_texture_30m
Cancelled: land_cover_30m
Cancelled: slope_30m
Cancelled: elevation_30m
Cancelled: rainfall_30m
Cancelled: twi_30m
Cancelled: drainage_density_30m
Cancelled: soil_texture_30m
Cancelled: land_cover_30m
Cancelled: slope_30m
Cancelled: elevation_30m
Cancelled: rainfall_30m
Cancelled: twi_30m
Cancelled: drainage_density_30m
Cancelled: soil_texture_30m
Cancelled: land_cover_30m
Cancelled: slope_30m


In [45]:
#Recreate clean exports
print(type(kano_boundary))
print(type(rainfall_clip))

<class 'ee.featurecollection.FeatureCollection'>
<class 'ee.image.Image'>


In [46]:
#Export all layers to Google Drive
folder_name = "Kano_Thematic_30m"

export_to_drive(rainfall_clip, "rainfall_30m", folder_name)
export_to_drive(elevation_clip, "elevation_30m", folder_name)
export_to_drive(slope_clip, "slope_30m", folder_name)
export_to_drive(land_cover_clip, "land_cover_30m", folder_name)
export_to_drive(soil_texture_clip, "soil_texture_30m", folder_name)
export_to_drive(drainage_density_clip, "drainage_density_30m", folder_name)
export_to_drive(twi_clip, "twi_30m", folder_name)

rainfall_30m export started.
elevation_30m export started.
slope_30m export started.
land_cover_30m export started.
soil_texture_30m export started.
drainage_density_30m export started.
twi_30m export started.


In [47]:
#Check the status of the export tasks
for task in ee.batch.Task.list():
    status = task.status()
    print(status['description'], ":", status['state'])

twi_30m : READY
drainage_density_30m : READY
soil_texture_30m : READY
land_cover_30m : READY
slope_30m : READY
elevation_30m : READY
rainfall_30m : READY
twi_30m : CANCELLED
drainage_density_30m : CANCELLED
soil_texture_30m : CANCELLED
land_cover_30m : CANCELLED
slope_30m : CANCELLED
elevation_30m : CANCELLED
rainfall_30m : CANCELLED
twi_30m : CANCELLED
drainage_density_30m : CANCELLED
soil_texture_30m : CANCELLED
land_cover_30m : CANCELLED
slope_30m : CANCELLED
elevation_30m : CANCELLED
rainfall_30m : CANCELLED
twi_30m : CANCELLED
drainage_density_30m : CANCELLED
soil_texture_30m : CANCELLED
land_cover_30m : CANCELLED
slope_30m : CANCELLED
elevation_30m : CANCELLED
rainfall_30m : CANCELLED
twi_30m : CANCELLED
drainage_density_30m : CANCELLED
soil_texture_30m : CANCELLED
land_cover_30m : CANCELLED
slope_30m : CANCELLED
elevation_30m : RUNNING
rainfall_30m : RUNNING
twi_30m : COMPLETED
drainage_density_30m : COMPLETED
soil_texture_30m : COMPLETED
land_cover_30m : COMPLETED
slope_30m : C